# Session 7 — Google Earth Engine I: collections, filters and reducers (CHIRPS rainfall)
*Séance 7 — Google Earth Engine I : collections, filtres et réducteurs (pluie CHIRPS)*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/ML4EO_Bootcamp/blob/main/notebooks/07_gee_chirps.ipynb)

**Course:** Machine Learning for Earth Observation (45 h) · **Instructor:** Noé Carème Fouotsa Manfouo

**By the end of this session you will be able to:**

1. Explain the client–server model of Earth Engine (lazy computation, `getInfo()`).
2. Filter an ImageCollection by date and area and reduce it over time and space.
3. Build a monthly rainfall time series for any polygon and export it as a table.
4. Automate the extraction for several administrative units.

**How to use this notebook:** read the lecture cells, run every code cell (`Shift + Enter`), then complete the cells marked
`# YOUR CODE HERE` in the lab section. Run the ✅ check cells to get instant feedback.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
import sys, pathlib, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("ML4EO_Bootcamp").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/ML4EO_Bootcamp.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *"geopandas rasterio rioxarray".split()], check=True)
    ROOT = pathlib.Path("ML4EO_Bootcamp")
else:
    ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data"
sys.path.append(str(ROOT / "notebooks"))   # course helper module (ml4eo_data.py)


def check(condition, message="Check failed"):
    """Instant feedback: raises an error with a hint if the condition is False."""
    assert condition, message
    print("✅ Passed")


def check_equal(value, expected, message=None):
    assert value == expected, message or f"Expected {expected!r}, got {value!r}"
    print("✅ Passed")

### Connect to Google Earth Engine
Since 13 November 2024, Earth Engine requires a **Google Cloud project** registered for Earth Engine (free for noncommercial use).
One-time setup: go to <https://code.earthengine.google.com/register>, choose *Unpaid usage → Academia & Research*, and create a project
(e.g. `ee-yourname`). Then write its ID below.

In [ ]:
import ee, geemap

EE_PROJECT = "ee-yourname"   # <-- replace with YOUR Cloud project ID

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()          # opens a sign-in link the first time
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready:", ee.String("Hello from Earth Engine").getInfo())

## Part A — Lecture

### How Earth Engine works
Your notebook only **describes** a computation (an object graph). Nothing is computed until you ask for a result with
`getInfo()`, a map display or an export — then Google's servers run it on petabytes of data. Three rules follow:
1. Use Earth Engine methods (`ee.Number.add`, `ee.List.map`) inside server-side code, not Python loops or `if`.
2. Call `getInfo()` only on **small** results (a number, a short table).
3. For big outputs (images, long tables), use `Export` tasks.

**CHIRPS** (Climate Hazards Group InfraRed Precipitation with Station data) gives daily rainfall at ~5.5 km since 1981,
between 50°S and 50°N (`UCSB-CHG/CHIRPS/DAILY`, band `precipitation`, mm/day).

In [ ]:
import geopandas as gpd

WARD_FILES = {"ZA1011001": DATA / "SA_LEVEL4" / "ADM4_PCODE_ZA1011001.shp",   # Western Cape
              "ZA6372040": DATA / "SA_LEVEL4" / "ADM4_PCODE_ZA6372040.shp"}   # North West


def ward_fc(code):
    """Upload a ward polygon from the course repository to Earth Engine as a FeatureCollection."""
    return geemap.geopandas_to_ee(gpd.read_file(WARD_FILES[code])[["ADM4_PCODE", "geometry"]])


ward = ward_fc("ZA1011001")
print("Ward area (km²):", ward.geometry().area(maxError=1).divide(1e6).getInfo())

In [ ]:
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")
year = chirps.filterDate("2021-01-01", "2022-01-01")
print("Images in 2021:", year.size().getInfo())

annual_total = year.sum().clip(ward)
Map = geemap.Map()
Map.centerObject(ward, 10)
Map.addLayer(year.sum(), {"min": 0, "max": 1200, "palette": ["white", "skyblue", "blue", "darkblue"]}, "Rainfall 2021 (mm)")
Map.addLayer(ward.style(color="red", fillColor="00000000"), {}, "Ward")
Map

### From images to a monthly time series
We build one image per month (sum of daily rainfall), then average it over the ward (`reduceRegion`).

In [ ]:
import pandas as pd


def monthly_rainfall(fc, start_year, end_year):
    """Monthly rainfall (mm) averaged over a FeatureCollection, as a pandas DataFrame."""
    months = ee.List.sequence(0, (end_year - start_year + 1) * 12 - 1)
    start = ee.Date.fromYMD(start_year, 1, 1)

    def one_month(n):
        d0 = start.advance(n, "month")
        total = chirps.filterDate(d0, d0.advance(1, "month")).sum()
        value = total.reduceRegion(reducer=ee.Reducer.mean(), geometry=fc.geometry(), scale=5566).get("precipitation")
        return ee.Feature(None, {"date": d0.format("YYYY-MM"), "rain_mm": value})

    table = ee.FeatureCollection(months.map(one_month))
    df = geemap.ee_to_df(table)
    df["date"] = pd.to_datetime(df["date"])
    return df.set_index("date")


rain = monthly_rainfall(ward, 2014, 2021)
rain.plot(figsize=(10, 3), legend=False, title="Monthly rainfall, ward ZA1011001 (mm)")

## Part B — Lab

**Task 1 — Annual totals.** From `rain`, compute the annual totals (`annual`, a Series indexed by year) and the driest
year (`driest_year`, an int).

In [ ]:
# YOUR CODE HERE
annual = None
driest_year = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(len(annual) == 8)
print("Driest year:", driest_year)

**Task 2 — Anomalies.** Compute the monthly **standardized anomaly** `z` = (value − mean of that calendar month) /
standard deviation of that calendar month, as a Series aligned with `rain`.

In [ ]:
# YOUR CODE HERE
z = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(abs(z.mean()) < 0.2, "Anomalies should average close to zero")

**Task 3 — Automate for several wards.** Loop over `WARD_FILES`, run `monthly_rainfall` for 2021 only, and build a
DataFrame `all_wards` with one column per ward code. Save it as `chirps_wards_2021.csv`.

In [ ]:
# YOUR CODE HERE
all_wards = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(list(all_wards.columns) == list(WARD_FILES))
check(len(all_wards) == 12)

**Export large tables.** For hundreds of wards, `getInfo()` becomes too slow; use an export task instead:
```python
task = ee.batch.Export.table.toDrive(collection=table, description="chirps_wards", fileFormat="CSV")
task.start()
```

**❓ Question —** Ward ZA1011001 is in the Western Cape. Does your driest year match the 2015–2018 Cape Town drought? What are CHIRPS's limits at 5.5 km resolution?

*Your answer here.*

## Summary

- Earth Engine computes lazily on the server; bring back only small results.
- filterDate / filterBounds → reduce over time (sum, mean) → reduce over space (reduceRegion).
- Wrap the workflow in a function to scale to many units; export big outputs.

**To go further:**

- Funk, C. et al. (2015). The climate hazards infrared precipitation with stations. *Scientific Data*, 2, 150066.
- Earth Engine Python API guide: https://developers.google.com/earth-engine/guides/python_install

---
*Adapted from the 2022 ML4EO bootcamp notebook 'CHIRPS rainfall data collection for South Africa' (N. C. Fouotsa Manfouo).*